In [ ]:
#pip install langchain langchain-huggingface transformersaccelerate torch

In [1]:
from transformers import pipeline

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware

from langchain_huggingface import (
    HuggingFacePipeline,
    ChatHuggingFace,
)

from langchain_core.tools import tool

In [ ]:
@tool
def get_weather(location: str) -> str:
    """Get the current weather."""

    if location.lower() == "munich":
        return "18°C and sunny"

    return "Unknown"


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""

    return a * b


tools = [
    get_weather,
    multiply,
]
systemPrompt = """
You are a helpful assistant.

Use tools whenever they are useful.
"""

pipe = pipeline(
    task="text-generation",
    model="Qwen/Qwen3-4B-Instruct",
    #model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    max_new_tokens=128,
)

hf_pipeline = HuggingFacePipeline(
    pipeline=pipe
)

llm = ChatHuggingFace(
    llm=hf_pipeline
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=systemPrompt,
    middleware=[
        TrajectoryLoggerMiddleware()
    ]
)

NameError: name 'tool' is not defined

In [ ]:
class TrajectoryLoggerMiddleware(AgentMiddleware):

    def before_model(self, state, runtime):
        print("\n================ MODEL INPUT =================")
        print(state["messages"][-1])

    def after_model(self, state, runtime):
        print("\n================ MODEL OUTPUT =================")
        print(state["messages"][-1])

    def before_tool(self, tool_call, runtime):
        print("\n================ TOOL CALL =================")
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")

    def after_tool(self, tool_call, result, runtime):
        print("\n================ TOOL RESULT =================")
        print(result)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            (
                "user",
                "What is 25 multiplied by 4?"
            )
        ]
    }
)

In [ ]:
print("\n================ RAW RESPONSE =================")
print(response)

print("\n================ FINAL ANSWER =================")
print(response["messages"][-1].content)

In [ ]:
print("\n================ TRAJECTORY =================")

for i, msg in enumerate(response["messages"]):

    print(f"\nStep {i}")
    print(f"Type: {msg.__class__.__name__}")

    print(msg.content)

    if hasattr(msg, "tool_calls"):
        print("Tool Calls:")
        print(msg.tool_calls)